# Cascading flow

PRMS cascades route Hortonian surface runoff and soil-zone flow (interflow
and Dunnian flow) from an HRU to its downslope HRUs or to a stream segment,
instead of sending all of it straight to the stream. This notebook runs the
Sagehen Creek domain with HRU cascades on, maps where the cascading water
goes, then runs the same model with cascades off and compares.
Groundwater cascades are not used here.

In [ ]:
import pathlib as pl
from pprint import pprint

import pywatershed as pws

## Control and parameters

The `sagehen_no_gw_cascades` control has `cascade_flag = 1` and
`cascadegw_flag = 0`. Output goes under `11_cascading_flow/`.

In [ ]:
output_dir = pl.Path("./11_cascading_flow/output")
domain_dir = pl.Path("../test_data/sagehen_5yr/")

In [ ]:
control_file = domain_dir / "sagehen_no_gw_cascades.control"
control = pws.Control.load_prms(control_file)
control.options["parameter_file"] = (
    domain_dir / control.options["parameter_file"][2:]
)
control.options["input_dir"] = domain_dir
control.options["imbalance_behavior"] = None
control.options["netcdf_output_dir"] = output_dir
control.options["netcdf_output_var_names"] = [
    *pws.PRMSRunoffCascadesNoDprst.get_variables(),
    *pws.PRMSSoilzoneCascadesNoDprst.get_variables(),
    *pws.PRMSGroundwaterNoDprst.get_variables(),
]

In [ ]:
parameters = pws.parameters.PrmsParameters.load(
    control.options["parameter_file"]
)

In [ ]:
# Derive the cascade parameters (hru_route_order, hru_down, ...) from the
# PRMS parameter file. PRMSRunoffCascadesNoDprst and
# PRMSSoilzoneCascadesNoDprst do this themselves when hru_route_order is
# missing, so this step is optional. Calling it explicitly is useful to
# inspect or save the derived parameters, or to preprocess once and share
# the result between the two cascade processes rather than have each
# derive it.
parameters = pws.utils.preprocess_cascades.preprocess_cascade_params(
    control, parameters
)

## The cascade model

### Model setup and run
`PRMSRunoffCascadesNoDprst` and `PRMSSoilzoneCascadesNoDprst` are the
cascade variants of runoff and soilzone; the other processes are unchanged.

In [ ]:
dis_hru = pws.Parameters.from_netcdf(
    domain_dir / "parameters_dis_hru.nc", encoding=False
)

process_dict = {
    "solar": pws.PRMSSolarGeometry,
    "atmosphere": pws.PRMSAtmosphere,
    "canopy": pws.PRMSCanopy,
    "snow": pws.PRMSSnow,
    "runoff": pws.PRMSRunoffCascadesNoDprst,
    "soilzone": pws.PRMSSoilzoneCascadesNoDprst,
    "groundwater": pws.PRMSGroundwaterNoDprst,
}

model_dict = {
    "control": control,
    "dis_hru": dis_hru,
    "model_order": list(process_dict.keys()),
}

for kk, vv in process_dict.items():
    model_dict[kk] = {"class": vv, "parameters": parameters, "dis": "dis_hru"}

In [ ]:
pprint(model_dict, sort_dicts=False)

In [ ]:
model = pws.Model(model_dict)

In [ ]:
model.run(netcdf_dir=output_dir, finalize=True)

### Time-mean maps

We examine the mean flow each HRU passes downslope: `hru_horton_cascflow`
(Hortonian surface runoff) and `hru_sz_cascadeflow` (interflow and Dunnian
flow).

In [ ]:
import xarray as xr

# load_dataarray reads and closes the file (open_dataarray would keep it
# open, blocking a later rerun of the model from overwriting it)
horton_casc = xr.load_dataarray(output_dir / "hru_horton_cascflow.nc")
sz_casc = xr.load_dataarray(output_dir / "hru_sz_cascadeflow.nc")

In [ ]:
# The sagehen HRU polygons are the "HRU" layer of the Sagehen.gdb
# geodatabase in the pywatershed GIS files (pws.utils.gis_files.download()).
proc_plot = pws.analysis.process_plot.ProcessPlot(
    pws.utils.get_gis_dir("sagehen_mf6"),
    hru_shp_file_name="Sagehen.gdb",
    hru_layer="HRU",
    seg_shp_file_name=None,
)

# Time-mean Hortonian surface runoff cascading to downslope HRUs
proc_plot.plot_hru_var(
    var_name="hru_horton_cascflow",
    process=model.processes["runoff"],
    data=horton_casc.mean(dim="time"),
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
)

In [ ]:
# Time-mean soilzone (interflow and Dunnian) flow cascading downslope
proc_plot.plot_hru_var(
    var_name="hru_sz_cascadeflow",
    process=model.processes["soilzone"],
    data=sz_casc.mean(dim="time"),
    data_units=sz_casc.attrs["units"],
    nhm_id=sz_casc["nhm_id"],
)

### Monthly map time series

In [ ]:
import panel as pn

pn.extension()

# Through time, one map per month with a play/pause "scrubber" widget.
# Hortonian cascade flow happens on a few melt or rain days, so the monthly
# maximum shows where and when; the mean would average it away. On this
# domain it amounts to two events in five years. The color scale is fixed
# across months so they compare. .embed() puts all frames in the output so
# playback runs in the browser.
horton_monthly = horton_casc.resample(time="MS").max()
horton_maps = proc_plot.plot_hru_var(
    var_name="hru_horton_cascflow",
    process=model.processes["runoff"],
    data=horton_monthly,
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
    clim=(0.0, float(horton_monthly.max())),
)
pn.panel(horton_maps, widget_type="scrubber", widget_location="bottom").embed()

In [ ]:
# Soilzone cascade flow (interflow and Dunnian) is sustained over weeks,
# so the monthly mean is the right summary here.
sz_monthly = sz_casc.resample(time="MS").mean()
sz_maps = proc_plot.plot_hru_var(
    var_name="hru_sz_cascadeflow",
    process=model.processes["soilzone"],
    data=sz_monthly,
    data_units=sz_casc.attrs["units"],
    nhm_id=sz_casc["nhm_id"],
    clim=(0.0, float(sz_monthly.max())),
)
pn.panel(sz_maps, widget_type="scrubber", widget_location="bottom").embed()

## The model without cascades
### Model setup and run
`PRMSRunoffNoDprst` and `PRMSSoilzoneNoDprst` replace the cascade classes.
The `sagehen_no_cascades` control is identical except `cascade_flag = 0`.
The extra cascade parameters derived above are ignored by the non-cascade
processes, so the same parameters object serves both models.

In [ ]:
output_dir_nc = pl.Path("./11_cascading_flow/output_no_cascades")
control_nc = pws.Control.load_prms(domain_dir / "sagehen_no_cascades.control")
control_nc.options["parameter_file"] = control.options["parameter_file"]
control_nc.options["input_dir"] = domain_dir
control_nc.options["imbalance_behavior"] = None
control_nc.options["netcdf_output_dir"] = output_dir_nc
control_nc.options["netcdf_output_var_names"] = [
    *pws.PRMSRunoffNoDprst.get_variables(),
    *pws.PRMSSoilzoneNoDprst.get_variables(),
    *pws.PRMSGroundwaterNoDprst.get_variables(),
]

process_dict_nc = {
    **process_dict,
    "runoff": pws.PRMSRunoffNoDprst,
    "soilzone": pws.PRMSSoilzoneNoDprst,
}
model_dict_nc = {
    "control": control_nc,
    "dis_hru": dis_hru,
    "model_order": list(process_dict_nc.keys()),
}
for kk, vv in process_dict_nc.items():
    model_dict_nc[kk] = {
        "class": vv,
        "parameters": parameters,
        "dis": "dis_hru",
    }

model_nc = pws.Model(model_dict_nc)
model_nc.run(netcdf_dir=output_dir_nc, finalize=True)

## Compare cascade and no-cascade results

### Time-mean maps of diffs (cascade - no_cascade)

Cascading moves surface runoff and soil-zone flow from upslope HRUs into
downslope ones, so the downslope HRUs infiltrate, store and evaporate more
water and send less runoff straight to the stream.

In [ ]:
def load_both(var_name):
    casc = xr.load_dataarray(output_dir / f"{var_name}.nc")
    no_casc = xr.load_dataarray(output_dir_nc / f"{var_name}.nc")
    return casc, no_casc


def diff_map(var_name):
    casc, no_casc = load_both(var_name)
    diff = (casc - no_casc).mean(dim="time")
    lim = float(abs(diff).max())
    return proc_plot.plot_hru_var(
        var_name=f"{var_name} (cascade - no cascade)",
        process=model.processes["runoff"],
        data=diff,
        data_units=casc.attrs["units"],
        nhm_id=casc["nhm_id"],
        clim=(-lim, lim),
        cmap="RdBu_r",
    )

In [ ]:
# Surface runoff reaching the stream
diff_map("sroff")

In [ ]:
# Interflow from the gravity reservoir
diff_map("ssres_flow")

In [ ]:
# Actual evapotranspiration
diff_map("hru_actet")

### Basin-total lateral flow time series

Cascading changes where water enters the stream network, and how much: the
water an HRU passes downslope can be stored, evaporated or infiltrated to
groundwater there instead. There is no channel in this model, so the flow
reaching the streams is summed over HRUs: surface runoff, interflow,
Dunnian flow and groundwater flow, area-weighted to a basin depth.

In [ ]:
import pandas as pd

hru_area = xr.DataArray(
    parameters.parameters["hru_area"],
    dims=["nhm_id"],
    coords={"nhm_id": parameters.parameters["nhm_id"]},
)


def basin_mean(da):
    return (da * hru_area).sum(dim="nhm_id") / hru_area.sum()


lateral_vars = ["sroff", "ssres_flow", "dunnian_flow", "gwres_flow"]
casc_tot = 0.0
no_casc_tot = 0.0
for vv in lateral_vars:
    casc, no_casc = load_both(vv)
    casc_tot = casc_tot + basin_mean(casc)
    no_casc_tot = no_casc_tot + basin_mean(no_casc)

basin_df = pd.DataFrame(
    {"cascade": casc_tot, "no cascade": no_casc_tot},
    index=casc["time"].values,
)
basin_df.hvplot.line(
    ylabel="inches/day",
    title="Basin lateral flow to streams, area-weighted",
    frame_width=800,
)